# Machine Learning

**Tarefa:** _[Classificação / Regressão / Clustering]_  
**Dataset:** _[nome / fonte]_  
**Objetivo:** _[o que o modelo deve prever ou descobrir?]_

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score)
import shap
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42

## 2. Carregamento e Visão Geral

In [ ]:
df = pd.read_parquet('../data/processed/arquivo_clean.parquet')
print(df.shape)
df.head()

## 3. Separação Features / Target

In [ ]:
TARGET = 'target'  # <- altere para o nome da coluna alvo

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')
print(f'Distribuição target (treino):\n{y_train.value_counts(normalize=True).round(3)}')

## 4. Pipeline e Modelo

In [ ]:
from sklearn.ensemble import RandomForestClassifier  # troque pelo modelo desejado

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=100, random_state=SEED))
])

# Validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'ROC-AUC CV: {scores.mean():.4f} ± {scores.std():.4f}')

## 5. Treino e Avaliação no Teste

In [ ]:
pipe.fit(X_train, y_train)
y_pred  = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'ROC-AUC Teste: {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot(cmap='Blues')
plt.title('Matriz de Confusão')
plt.show()

## 6. Importância de Features

In [ ]:
importances = pd.Series(
    pipe['model'].feature_importances_, index=X.columns
).sort_values(ascending=False).head(15)

importances.plot(kind='barh', figsize=(8, 5))
plt.title('Top 15 Features por Importância')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Explicabilidade (SHAP)

In [ ]:
explainer   = shap.TreeExplainer(pipe['model'])
shap_values = explainer(X_test)
shap.plots.beeswarm(shap_values)

## 8. Conclusões

- _Performance geral_
- _Features mais relevantes_
- _Limitações e melhorias_